# Classification Modeling

Use this notebook for labeled classification tasks on structured data. It covers binary and multiclass prediction with baseline models, random forest, XGBoost, and an optional RNN scaffold for true sequence data.


## Recommended Modeling Mindset

- Start with a simple baseline before a complex model
- Keep preprocessing inside a reproducible pipeline
- Compare models on the metric that matches the business goal
- Use interpretability tools for stakeholder-facing narratives
- Do not use RNNs for ordinary tabular data unless each observation is a true ordered sequence


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

pd.set_option('display.max_columns', None)
plt.style.use('ggplot')


In [ ]:
DATA_PATH = Path('data.csv')

REGRESSION_TARGET = 'order_value'
BINARY_TARGET = 'converted'
MULTICLASS_TARGET = 'segment'
DROP_COLUMNS = ['customer_id']

df = pd.read_csv(DATA_PATH)
display(df.head())
print(df.shape)


In [ ]:
def build_preprocessor(feature_df):
    numeric_features = feature_df.select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in feature_df.columns if c not in numeric_features]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                'num',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                'cat',
                Pipeline([
                    ('imputer', SimpleImputer(strategy='most_frequent')),
                    ('encoder', OneHotEncoder(handle_unknown='ignore')),
                ]),
                categorical_features,
            ),
        ],
        remainder='drop',
    )
    return preprocessor, numeric_features, categorical_features


def prepare_xy(df, target_col):
    model_df = df.dropna(subset=[target_col]).copy()
    feature_cols = [c for c in model_df.columns if c not in set(DROP_COLUMNS + [target_col])]
    X = model_df[feature_cols].copy()
    y = model_df[target_col].copy()
    return X, y


## 2. Binary Classification

Use this when the target is yes/no, such as converted vs not converted, churn vs retained, fraud vs non-fraud.

Suggested metrics:
- ROC AUC when ranking quality matters
- F1 when classes are imbalanced and positive detection matters
- Accuracy only when classes are reasonably balanced


In [ ]:
X_bin, y_bin = prepare_xy(df, BINARY_TARGET)
preprocessor_bin, _, _ = build_preprocessor(X_bin)

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

binary_models = {
    'logistic_regression': LogisticRegression(max_iter=1000),
    'random_forest': RandomForestClassifier(n_estimators=300, random_state=42),
}

if XGBOOST_AVAILABLE:
    binary_models['xgboost'] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='logloss',
        random_state=42,
    )

binary_results = []

for name, estimator in binary_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor_bin),
        ('model', estimator),
    ])
    pipeline.fit(X_train_bin, y_train_bin)
    preds = pipeline.predict(X_test_bin)
    if hasattr(pipeline.named_steps['model'], 'predict_proba'):
        probs = pipeline.predict_proba(X_test_bin)[:, 1]
        roc_auc = roc_auc_score(y_test_bin, probs)
    else:
        roc_auc = np.nan
    binary_results.append({
        'model': name,
        'accuracy': accuracy_score(y_test_bin, preds),
        'f1': f1_score(y_test_bin, preds),
        'roc_auc': roc_auc,
    })

display(pd.DataFrame(binary_results).sort_values('roc_auc', ascending=False))


### Optional: RNN for Sequential Binary Prediction

Only use this if each observation is a sequence, such as customer event streams, sensor readings, or time-ordered click sequences. For ordinary tabular data, logistic regression or tree models are usually better baselines.


In [ ]:
# Example scaffold only. This section will need sequence-shaped tensors.
try:
    from tensorflow import keras
    from tensorflow.keras import layers

    print('TensorFlow available. Replace X_seq_train, y_seq_train, X_seq_test with your sequence tensors.')
    print('Suggested architecture: Masking -> LSTM -> Dense(sigmoid)')
except ImportError:
    print('Install tensorflow to use the RNN section.')


## 3. Multiclass Classification

Use this when the target has more than two categories, such as segment, issue type, product tier, or intent bucket.

Suggested metrics:
- Macro F1 when class balance matters
- Weighted F1 when class imbalance is strong
- Accuracy as a secondary metric


In [ ]:
X_multi, y_multi = prepare_xy(df, MULTICLASS_TARGET)
preprocessor_multi, _, _ = build_preprocessor(X_multi)

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

multiclass_models = {
    'multinomial_logistic': LogisticRegression(max_iter=1000, multi_class='multinomial'),
    'random_forest': RandomForestClassifier(n_estimators=300, random_state=42),
}

if XGBOOST_AVAILABLE:
    multiclass_models['xgboost'] = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric='mlogloss',
        random_state=42,
    )

multiclass_results = []

for name, estimator in multiclass_models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor_multi),
        ('model', estimator),
    ])
    pipeline.fit(X_train_multi, y_train_multi)
    preds = pipeline.predict(X_test_multi)
    multiclass_results.append({
        'model': name,
        'accuracy': accuracy_score(y_test_multi, preds),
        'macro_f1': f1_score(y_test_multi, preds, average='macro'),
        'weighted_f1': f1_score(y_test_multi, preds, average='weighted'),
    })

display(pd.DataFrame(multiclass_results).sort_values('macro_f1', ascending=False))


## Interpreting Results

For each chosen model, summarize:

- Why this model was selected
- Which metric made it the winner
- Which features matter most
- What business action the team could take
- What risks remain, such as leakage, imbalance, or weak sample size
